In [ ]:
import numpy as np
import torch
import time
import os
import csv
import matplotlib.pyplot as plt

from datasets import load_dataset, concatenate_datasets
from scipy.stats import pearsonr

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
train = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="train")
val = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="dev")
test = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="test")

print(train)

train_df = train.to_pandas()
print("Sample data (first 5 rows):")
print(train_df.head())


In [ ]:

full_data = concatenate_datasets([train, val, test])

len_total = len(full_data)

target_train = int(round(0.70 * len_total))
target_val   = int(round(0.20 * len_total))
target_test  = len_total - target_train - target_val

split_1 = full_data.train_test_split(train_size=target_train, seed=42)
train_data = split_1["train"]
remaining = split_1["test"]

split_2 = remaining.train_test_split(train_size=target_val, seed=42)
val_data = split_2["train"]
test_data = split_2["test"]

print("Train size:", len(train_data))
print("Validation size:", len(val_data))
print("Test size:", len(test_data))

In [ ]:
TEXT_COL = "text"
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess(example):
    encoded = tokenizer(
        example[TEXT_COL],
        truncation=True,
        max_length=256,
    )
    encoded["labels"] = [float(example[e]) for e in EMOTIONS]
    return encoded

train_tok = train_data.map(preprocess)
val_tok   = val_data.map(preprocess)
test_tok  = test_data.map(preprocess)

cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

In [ ]:
def safe_pearson(x, y):
    r, _ = pearsonr(x, y)
    return 0.0 if np.isnan(r) else float(r)

def compute_metrics(eval_pred):
    preds = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    print("preds shape:", preds.shape)
    print("labels shape:", labels.shape)

    metrics = {}
    rs = []

    for i, emo in enumerate(EMOTIONS):
        pred_col = preds[:, i]
        label_col = labels[:, i]

        r = safe_pearson(pred_col, label_col)
        metrics[f"pearson_{emo}"] = r
        rs.append(r)

    metrics["pearson_mean"] = float(np.mean(rs))
    return metrics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

LOG_FILE = "/content/drive/MyDrive/DistilBERT_Log.csv"
os.makedirs("/content/drive/MyDrive", exist_ok=True)

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch",
        "train_loss",
        "eval_loss",
        "pearson_mean",
        "pearson_anger",
        "pearson_fear",
        "pearson_joy",
        "pearson_sadness",
        "pearson_surprise"
    ])

epoch_list = []
train_loss_list = []
eval_loss_list = []
accuracy_list = []

class SaveMetricsCallback(TrainerCallback):
    def __init__(self, file_path):
        self.file_path = file_path
        self.current_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None:
            epoch = int(metrics.get("epoch", state.epoch))
            train_loss = self.current_train_loss if self.current_train_loss is not None else ""
            eval_loss = float(metrics.get("eval_loss", 0.0))
            accuracy = float(metrics.get("eval_pearson_mean", 0.0))

            pearson_anger = float(metrics.get("eval_pearson_anger", 0.0))
            pearson_fear = float(metrics.get("eval_pearson_fear", 0.0))
            pearson_joy = float(metrics.get("eval_pearson_joy", 0.0))
            pearson_sadness = float(metrics.get("eval_pearson_sadness", 0.0))
            pearson_surprise = float(metrics.get("eval_pearson_surprise", 0.0))

            epoch_list.append(epoch)
            train_loss_list.append(train_loss)
            eval_loss_list.append(eval_loss)
            accuracy_list.append(accuracy)

            with open(self.file_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    epoch,
                    train_loss,
                    eval_loss,
                    accuracy,
                    pearson_anger,
                    pearson_fear,
                    pearson_joy,
                    pearson_sadness,
                    pearson_surprise
                ])

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(EMOTIONS),
    problem_type="regression"
).to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/distilbert_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=100,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[SaveMetricsCallback(LOG_FILE)],
)


start = time.time()
trainer.train()
end = time.time()

print(f"Total training time: {end - start:.1f} seconds")
print(f"Average time per epoch: {(end - start) / training_args.num_train_epochs:.1f} seconds")
print("Log file saved at:", LOG_FILE)

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, train_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, eval_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, accuracy_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation Pearson Mean")
plt.title("Validation Pearson Mean vs Epoch")
plt.grid(True)
plt.show()

In [ ]:
test_results = trainer.evaluate(test_tok)
print("Test results:", test_results)

In [ ]:
def predict_intensities(text: str):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits.detach().cpu().numpy()[0]

    discrete = np.clip(np.rint(logits), 0, 3).astype(int)

    return {
        EMOTIONS[i]: {
            "raw": float(logits[i]),
            "intensity_0_3": int(discrete[i])
        }
        for i in range(len(EMOTIONS))
    }

print(predict_intensities("I feel so happy and joyful today!"))